In [ ]:
import cv2
import numpy as np
from scipy.spatial import distance as dist

# --- CONFIGURATION ---
TARGET_COLOR = {
    "lower": np.array([100, 150, 0]),
    "upper": np.array([140, 255, 255]),
}
MIN_AREA = 500  # Minimum pixel area to ignore noise

class CentroidTracker:
    # Add max_distance parameter (e.g., 50 pixels)
    def __init__(self, max_disappeared=50, max_distance=50):
        self.next_object_id = 0
        self.objects = {}       
        self.disappeared = {}   
        self.trajectories = {}  
        
        self.max_disappeared = max_disappeared
        self.max_distance = max_distance # Store the new parameter


    def register(self, centroid):
        # Assign a new ID to a new object and start its trajectory list
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0
        self.trajectories[self.next_object_id] = [centroid]
        self.next_object_id += 1

    def deregister(self, object_id):
        # Stop tracking an object if it's lost
        del self.objects[object_id]
        del self.disappeared[object_id]

    def update(self, input_centroids):
        # If no objects detected this frame, increment disappeared counts
        if len(input_centroids) == 0:
            for object_id in list(self.objects.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.objects

        # If we aren't tracking anything yet, register everything detected
        if len(self.objects) == 0:
            for i in range(len(input_centroids)):
                self.register(input_centroids[i])
        else:
            # We have old objects and new detections. Match them by distance.
            object_ids = list(self.objects.keys())
            object_centroids = list(self.objects.values())

            # Compute Euclidean distance between all old and new centroids
            D = dist.cdist(np.array(object_centroids), input_centroids)

            # Find the smallest distances to match old IDs to new detections
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            used_rows = set()
            used_cols = set()

            for row, col in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue
                
                # --- NEW DISTANCE CHECK ---
                # D[row, col] is the distance between the old and new centroid.
                # If it's greater than our threshold, ignore the match.
                if D[row, col] > self.max_distance:
                    continue
                # --------------------------
                
                # Update the object with its new centroid
                object_id = object_ids[row]
                self.objects[object_id] = input_centroids[col]
                self.disappeared[object_id] = 0
                self.trajectories[object_id].append(input_centroids[col])
                
                used_rows.add(row)
                used_cols.add(col)

            # Check for objects that went missing
            unused_rows = set(range(0, D.shape[0])).difference(used_rows)
            for row in unused_rows:
                object_id = object_ids[row]
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)

            # Register entirely new objects that just appeared
            unused_cols = set(range(0, D.shape[1])).difference(used_cols)
            for col in unused_cols:
                self.register(input_centroids[col])

        return self.objects

# --- MAIN LOOP ---
cap = cv2.VideoCapture("track test.mov")
tracker = CentroidTracker(max_disappeared=10, max_distance=800)

while True:
    ret, frame = cap.read()
    if not ret: break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, TARGET_COLOR["lower"], TARGET_COLOR["upper"])
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)

    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    current_centroids = []

    # Get centroids for ALL contours above a certain size
    for c in contours:
        if cv2.contourArea(c) > MIN_AREA:
            M = cv2.moments(c)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                current_centroids.append((cx, cy))
                
                # Draw contour boundaries
                cv2.drawContours(frame, [c], -1, (0, 255, 0), 2)

    # Feed the detected centroids into our tracker
    tracked_objects = tracker.update(current_centroids)

    # Draw the assigned IDs and tracking dots
    for object_id, centroid in tracked_objects.items():
        text = f"ID {object_id}"
        cv2.putText(frame, text, (centroid[0] - 10, centroid[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        cv2.circle(frame, (centroid[0], centroid[1]), 5, (0, 0, 255), -1)

    # Display fix
    scale = 0.5
    preview = cv2.resize(frame, (int(frame.shape[1] * scale), int(frame.shape[0] * scale)))
    cv2.imshow("Multi-Object Centroid Tracking", preview)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Retrieve your physics data:
# print("Trajectory for ID 0:", tracker.trajectories.get(0))


In [26]:
tracker.trajectories.get(1)

[(2830, 1757),
 (2833, 1752),
 (2833, 1755),
 (2833, 1755),
 (2832, 1756),
 (2832, 1756),
 (2833, 1757),
 (2833, 1755),
 (2841, 1760),
 (2828, 1765),
 (2827, 1761),
 (2822, 1755),
 (2824, 1761),
 (2844, 1768),
 (2825, 1737),
 (2836, 1324),
 (2838, 1322),
 (2833, 1324),
 (2836, 1323)]